Decision Tree Model

In [23]:
import numpy as np
import pandas as pd

# Load processed data
X_train = np.load("../data/processed/X_train_scaled.npy")
X_valid = np.load("../data/processed/X_valid_scaled.npy")
y_train = np.load("../data/processed/y_train.npy")
y_valid = np.load("../data/processed/y_valid.npy")
feature_names = pd.read_csv("../data/processed/feature_names.csv", header=None)[0].tolist()

In [29]:
# Import Decision Tree classification model and evaluation metrics to assess the model

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix, log_loss, make_scorer

# Initialize the baseline Decision Tree model with a defined random state for reproducible results. 

dt_baseline_model = DecisionTreeClassifier(random_state=42)

# Fit the model to the training data
dt_baseline_model.fit(X_train, y_train)

print("Baseline Decision Tree Model successfully initialized and trained on the training data.")

Baseline Decision Tree Model successfully initialized and trained on the training data.


In [3]:
# Import evaluation function that takes a specified model and test data as inputs, 
# uses the model to predict Gallstone Status and outputs evaluation metrics of the model.
# See code for evaluate_model function in evaluation_utility.py

from Utility_Functions.evaluation_utility import evaluate_model

evaluate_model(dt_baseline_model, X_valid, y_valid, model_name="Baseline Decision Tree Model")


Baseline Decision Tree Model Hyperparameters
  max_depth: None
  min_samples_leaf: 1
  min_samples_split: 2
  criterion: gini
  max_features: None

Baseline Decision Tree Model Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.78      0.75      0.77        24
   Gallstone       0.76      0.79      0.78        24

    accuracy                           0.77        48
   macro avg       0.77      0.77      0.77        48
weighted avg       0.77      0.77      0.77        48

Accuracy: 77.08%
AUC Score: 77.08%
Log Loss (Calibration): 8.2600
Specificity (True Negative Rate): 75.00%

Confusion Matrix (Raw Counts):
   Predicted 0  |  Predicted 1
Actual 0:  18   |  6
Actual 1:  5    |  19


##### Decision Tree Model Evaluation: Baseline Results

This analysis summarizes the performance of the initial, untuned Decision Tree Classifier on the test dataset (N=48 samples, perfectly balanced).

---

##### 1. Explanation of Evaluation Metrics

| Metric | Interpretation | Desired Value |
| :--- | :--- | :--- |
| **Accuracy** | Overall proportion of correct predictions (True Positives + True Negatives) out of all test cases. | Closer to 100% |
| **Precision** | Of all cases predicted as **Positive (Gallstones)**, how many were actually Gallstones? (Focuses on minimizing False Positives). | Closer to 1.0 |
| **Recall** | Of all cases that were **truly Positive (Gallstones)**, how many were correctly identified? (Focuses on minimizing False Negatives). | Closer to 1.0 |
| **F1-Score** | The harmonic mean of Precision and Recall. Provides a single measure that balances both metrics. | Closer to 1.0 |
| **AUC Score** | **Area Under the ROC Curve**. Measures the model's ability to distinguish between the two classes across all possible thresholds (not just the single 0.5 threshold used to get a final prediction). | Closer to 100% |
| **Specificity** | The True Negative Rate. Measures the proportion of cases correctly identified as **Negative (No Gallstones)**. | Closer to 100% |
| **Log Loss** | A measure of the error based on the model's predicted probabilities. Penalizes confident, incorrect predictions heavily. | Closer to 0.0 |

---

##### 2. Discussion of Baseline Results
 > **Context:** The Decision Tree first outputs a probability of Gallstone Status (0.0 to 1.0) and then converts this to a final **hard class prediction** (0 or 1) using the default threshold of 0.5.

| Metric | Result | Interpretation |
| :--- | :--- | :--- |
| **Accuracy** | **77.08%** | Solid baseline; significantly better than 50% chance. |
| **AUC Score** | **77.08%** | Strong discriminator; the probabilities are more reliable than the hard class predictions. |
| **F1-Score (0 & 1)** | **0.77** | Performance is perfectly balanced across both classes. |
| **Log Loss** | **8.26** | Low error rate, indicating the model's probability predictions are moderately well-calibrated. |
| **Specificity** | **75.00%** | The model correctly ruled out Gallstones 68.75% of the time. |
| **Confusion Matrix** | **TP: 19, TN: 18** | $19$ and $18$ samples were correctly classified in the two categories, while $5$ and $6$ samples were misclassified (FP: 6, FN: 5). |

The consistent performance across precision, recall, and specificity (≈ 75%) indicates that the model is likely underfitting the data — it has learned general patterns but lacks the complexity to capture finer distinctions between classes.

Since the Decision Tree was trained with default hyperparameters (e.g., max_depth=None, no pruning), it effectively grew until all leaves were pure or contained very few samples, which can lead to unstable generalization.

Nevertheless, the AUC score of 77.08% suggests that the model retains moderate discriminative ability. This implies that with appropriate hyperparameter tuning (e.g., constraining depth, adjusting min_samples_split, or applying cost-complexity pruning), the model’s hard classification metrics (accuracy, F1-score) could improve meaningfully.

---

##### 3. Next Steps: Model Fine-Tuning

The current focus shifts from validation to **optimization** through systematic hyperparameter search to maximize the model's generalization ability.

1.  **Objective:** The primary goal is to find the optimal balance between the model's complexity and its performance, typically by maximizing the **AUC Score**.

2.  **Hyperparameter Search:** Implement **`sklearn.model_selection.GridSearchCV`** to explore a systematic range of regularization parameters.

3.  **Key Parameters to Tune:**
    * **`max_depth`**: To allow the tree to learn more complex relationships.
    * **`min_samples_leaf`**: To control the size and noise tolerance of the resulting leaf nodes.
    * **`criterion`**: Compare the performance when using `'gini'` versus `'entropy'`.

4.  **Model Comparison:** Use the optimized Decision Tree to set a high benchmark for comparison against the other planned models, such as Logistic Regression and Random Forest.

In [4]:
# Baseline model results for comparison with Hyperparameter tuned models

evaluate_model(dt_baseline_model, X_valid, y_valid, model_name="Baseline Decision Tree Model")


Baseline Decision Tree Model Hyperparameters
  max_depth: None
  min_samples_leaf: 1
  min_samples_split: 2
  criterion: gini
  max_features: None

Baseline Decision Tree Model Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.78      0.75      0.77        24
   Gallstone       0.76      0.79      0.78        24

    accuracy                           0.77        48
   macro avg       0.77      0.77      0.77        48
weighted avg       0.77      0.77      0.77        48

Accuracy: 77.08%
AUC Score: 77.08%
Log Loss (Calibration): 8.2600
Specificity (True Negative Rate): 75.00%

Confusion Matrix (Raw Counts):
   Predicted 0  |  Predicted 1
Actual 0:  18   |  6
Actual 1:  5    |  19


In [6]:
# Import hyperparameter tuning and evaluation function from file
# See code for hyperparameter tuning function in model_tuning_utility.py

from Utility_Functions.model_tuning_utility import optimize_tree_model
from Utility_Functions.evaluation_utility import evaluate_model

# 1st iteration of Hyperparameter tuned model using 3 folds

dt_tuned_model_1 = optimize_tree_model('DT', n_splits=3, X_train=X_train, y_train=y_train, n_repeats=1, n_iter_search=1, scoring='auc')

evaluate_model(dt_tuned_model_1, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=3)")


Starting Hyperparameter Search for Decision Tree with 3 evaluations...
Search Method: Grid Search
CV Strategy: Stratified KFold (K=3)
Optimizing for: AUC Score
Fitting 3 folds for each of 52920 candidates, totalling 158760 fits
Grid Search with 3 evaluations complete.

Decision Tree Results
Best AUC Score found: 0.7853

Hyperparameter tuned Decision Tree Model (CV=3) Hyperparameters
  max_depth: 4
  min_samples_leaf: 20
  min_samples_split: 2
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=3) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.68      0.71      0.69        24
   Gallstone       0.70      0.67      0.68        24

    accuracy                           0.69        48
   macro avg       0.69      0.69      0.69        48
weighted avg       0.69      0.69      0.69        48

Accuracy: 68.75%
AUC Score: 78.21%
Log Loss (Calibration): 0.5190
Specificity (True Negative Rate): 7

In [4]:
# 2nd iteration of Hyperparameter tuned model using 5 folds

dt_tuned_model_2 = optimize_tree_model('DT', n_splits=5, X_train=X_train, y_train=y_train, n_repeats=1, n_iter_search=1, scoring='auc')

evaluate_model(dt_tuned_model_2, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=5)")


Starting Hyperparameter Search for Decision Tree with 5 evaluations...
Search Method: Grid Search
CV Strategy: Stratified KFold (K=5)
Optimizing for: AUC Score
Fitting 5 folds for each of 45360 candidates, totalling 226800 fits
Grid Search with 5 evaluations complete.

Decision Tree Results
Best AUC Score found: 0.7298

Hyperparameter tuned Decision Tree Model (CV=5) Hyperparameters
  max_depth: 7
  min_samples_leaf: 5
  min_samples_split: 2
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=5) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.75      0.75      0.75        24
   Gallstone       0.75      0.75      0.75        24

    accuracy                           0.75        48
   macro avg       0.75      0.75      0.75        48
weighted avg       0.75      0.75      0.75        48

Accuracy: 75.00%
AUC Score: 77.52%
Log Loss (Calibration): 1.2980
Specificity (True Negative Rate): 75

In [ ]:
# 2nd iteration of Hyperparameter tuned model using 5 folds, without post pruning
from Utility_Functions.model_tuning_utility import optimize_tree_model

dt_tuned_model_2 = optimize_tree_model('DT', n_splits=5, X_train=X_train, y_train=y_train, n_repeats=1, n_iter_search=1, scoring='auc')

evaluate_model(dt_tuned_model_2, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=5)")


Starting Hyperparameter Search for Decision Tree with 5 evaluations...
Search Method: Grid Search
CV Strategy: Stratified KFold (K=5)
Optimizing for: AUC Score
Fitting 5 folds for each of 1764 candidates, totalling 8820 fits
Grid Search with 5 evaluations complete.

Decision Tree Results
Best AUC Score found: 0.7827

Hyperparameter tuned Decision Tree Model (CV=5) Hyperparameters
  max_depth: 5
  min_samples_leaf: 5
  min_samples_split: 20
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=5) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.73      0.79      0.76        24
   Gallstone       0.77      0.71      0.74        24

    accuracy                           0.75        48
   macro avg       0.75      0.75      0.75        48
weighted avg       0.75      0.75      0.75        48

Accuracy: 75.00%
AUC Score: 76.39%
Log Loss (Calibration): 0.6210
Specificity (True Negative Rate): 79.1

In [ ]:
# 3rd iteration of Hyperparameter tuned model using 10 folds 

dt_tuned_model_3 = optimize_tree_model('DT', n_splits=10, X_train=X_train, y_train=y_train, n_repeats=1, n_iter_search=1, scoring='auc')

evaluate_model(dt_tuned_model_3, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=10)")


Starting Hyperparameter Search for Decision Tree with 10 evaluations...
Search Method: Grid Search
CV Strategy: Stratified KFold (K=10)
Optimizing for: AUC Score
Fitting 10 folds for each of 45360 candidates, totalling 453600 fits
Grid Search with 10 evaluations complete.

Decision Tree Results
Best AUC Score found: 0.7170

Hyperparameter tuned Decision Tree Model (CV=10) Hyperparameters
  max_depth: 7
  min_samples_leaf: 5
  min_samples_split: 20
  criterion: gini
  max_features: log2

Hyperparameter tuned Decision Tree Model (CV=10) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.73      0.92      0.81        24
   Gallstone       0.89      0.67      0.76        24

    accuracy                           0.79        48
   macro avg       0.81      0.79      0.79        48
weighted avg       0.81      0.79      0.79        48

Accuracy: 79.17%
AUC Score: 79.77%
Log Loss (Calibration): 0.5116
Specificity (True Negative Ra

In [6]:
# 4th iteration of Hyperparameter tuned model using 3 folds with 3 repeats (9 evaluations)

dt_tuned_model_4 = optimize_tree_model('DT', n_splits=3, X_train=X_train, y_train=y_train, n_repeats=3, n_iter_search=1, scoring='balanced_accuracy')

evaluate_model(dt_tuned_model_4, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=3, 3 repeats)")


Starting Hyperparameter Search for Decision Tree with 9 evaluations...
Search Method: Grid Search
CV Strategy: Repeated Stratified KFold (K=3, R=3)
Optimizing for: Balanced Accuracy
Fitting 9 folds for each of 45360 candidates, totalling 408240 fits
Grid Search with 9 evaluations complete.

Decision Tree Results
Best Balanced Accuracy found: 0.7248

Hyperparameter tuned Decision Tree Model (CV=3, 3 repeats) Hyperparameters
  max_depth: 3
  min_samples_leaf: 20
  min_samples_split: 2
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=3, 3 repeats) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.72      0.88      0.79        24
   Gallstone       0.84      0.67      0.74        24

    accuracy                           0.77        48
   macro avg       0.78      0.77      0.77        48
weighted avg       0.78      0.77      0.77        48

Accuracy: 77.08%
AUC Score: 79.77%
Log Loss (Cali

In [ ]:
# 5th iteration of Hyperparameter tuned model using 3 folds with 5 repeats (15 evaluations)

dt_tuned_model_5 = optimize_tree_model('DT', n_splits=3, X_train=X_train, y_train=y_train, n_repeats=5, n_iter_search=1, scoring='balanced_accuracy')

evaluate_model(dt_tuned_model_5, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model v2 (CV=3, 5 repeats)")


Starting Hyperparameter Search for Decision Tree with 15 evaluations...
Search Method: Grid Search
CV Strategy: Repeated Stratified KFold (K=3, R=5)
Optimizing for: Balanced Accuracy
Fitting 15 folds for each of 45360 candidates, totalling 680400 fits
Grid Search with 15 evaluations complete.

Decision Tree Results
Best Balanced Accuracy found: 0.7218

Hyperparameter tuned Decision Tree Model v2 (CV=3, 5 repeats) Hyperparameters
  max_depth: 3
  min_samples_leaf: 50
  min_samples_split: 2
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model v2 (CV=3, 5 repeats) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.69      0.92      0.79        24
   Gallstone       0.88      0.58      0.70        24

    accuracy                           0.75        48
   macro avg       0.78      0.75      0.74        48
weighted avg       0.78      0.75      0.74        48

Accuracy: 75.00%
AUC Score: 79.34%
Log L

In [34]:
# 5th iteration of Hyperparameter tuned model using 3 folds with 5 repeats (15 evaluations)

dt_tuned_model_5_1 = optimize_tree_model('DT', n_splits=3, X_train=X_train_selected, y_train=y_train_selected, n_repeats=5, n_iter_search=1, scoring='balanced_accuracy')

evaluate_model(dt_tuned_model_5_1, X_valid_selected, y_valid_selected, model_name="Hyperparameter tuned Decision Tree Model v2 (CV=3, 5 repeats)")


Starting Hyperparameter Search for Decision Tree with 15 evaluations...
Search Method: Grid Search
CV Strategy: Repeated Stratified KFold (K=3, R=5)
Optimizing for: Balanced Accuracy
Fitting 15 folds for each of 1764 candidates, totalling 26460 fits
Grid Search with 15 evaluations complete.

Decision Tree Results
Best Balanced Accuracy found: 0.7276

Hyperparameter tuned Decision Tree Model v2 (CV=3, 5 repeats) Hyperparameters
  max_depth: 4
  min_samples_leaf: 1
  min_samples_split: 20
  criterion: gini
  max_features: sqrt

Hyperparameter tuned Decision Tree Model v2 (CV=3, 5 repeats) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.71      0.92      0.80        24
   Gallstone       0.88      0.62      0.73        24

    accuracy                           0.77        48
   macro avg       0.80      0.77      0.77        48
weighted avg       0.80      0.77      0.77        48

Accuracy: 77.08%
AUC Score: 79.77%
Log Los

In [9]:
# 6th iteration of Hyperparameter tuned model using 5 folds with 5 repeats (25 evaluations)

dt_tuned_model_6 = optimize_tree_model('DT', n_splits=5, X_train=X_train, y_train=y_train, n_repeats=5, n_iter_search=1, scoring='balanced_accuracy')

evaluate_model(dt_tuned_model_6, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=5, 5 repeats)")


Starting Hyperparameter Search for Decision Tree with 25 evaluations...
Search Method: Grid Search
CV Strategy: Repeated Stratified KFold (K=5, R=5)
Optimizing for: Balanced Accuracy
Fitting 25 folds for each of 45360 candidates, totalling 1134000 fits
Grid Search with 25 evaluations complete.

Decision Tree Results
Best Balanced Accuracy found: 0.7144

Hyperparameter tuned Decision Tree Model (CV=5, 5 repeats) Hyperparameters
  max_depth: 3
  min_samples_leaf: 5
  min_samples_split: 20
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=5, 5 repeats) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.68      0.88      0.76        24
   Gallstone       0.82      0.58      0.68        24

    accuracy                           0.73        48
   macro avg       0.75      0.73      0.72        48
weighted avg       0.75      0.73      0.72        48

Accuracy: 72.92%
AUC Score: 80.64%
Log Loss (

In [36]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# Assuming X_train, y_train, and feature_names are already loaded
# IMPORTANT: Ensure feature_names is the cleaned, 38-element list!

# Initialize and train the Decision Tree model
dt_model = DecisionTreeClassifier(random_state=42) 
dt_model.fit(X_train, y_train)

# Get the feature importances
importances = dt_model.feature_importances_

# Create a DataFrame for sorting and viewing
feature_importance_df_dt = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Top 10 Features by Decision Tree Importance:")
print(feature_importance_df_dt.head(10))

Top 10 Features by Decision Tree Importance:
                         Feature  Importance
35        C-Reactive_Protein_CRP    0.238987
22         Visceral_Fat_Area_VFA    0.098987
37                     Vitamin_D    0.078555
11       Extracellular_Water_ECW    0.075605
33                    Creatinine    0.067886
9            Body_Mass_Index_BMI    0.065064
27   Low_Density_Lipoprotein_LDL    0.059377
25                       Glucose    0.047841
28  High_Density_Lipoprotein_HDL    0.042070
18                  Bone_Mass_BM    0.041554


In [39]:
from sklearn.feature_selection import SelectFromModel

# Initialize the selector using the trained DT model
# We'll use the 'median' threshold again as an example
sfm_dt = SelectFromModel(dt_model, threshold='mean', prefit=True) 

# Transform the data to keep only the selected features
X_train_selected_dt = sfm_dt.transform(X_train)
X_valid_selected_dt = sfm_dt.transform(X_valid)

# Get the names of the selected features
selected_indices_dt = sfm_dt.get_support(indices=True)
selected_feature_names_dt = [feature_names[i] for i in selected_indices_dt]

print(f"\nSelected features using DT and SelectFromModel: {len(selected_feature_names_dt)}")
print(selected_feature_names_dt)
print(f"New X_train shape: {X_train_selected_dt.shape}")


Selected features using DT and SelectFromModel: 12
['Body_Mass_Index_BMI', 'Extracellular_Water_ECW', 'Extracellular_Fluid_Total_Body_Water_ECF_TBW', 'Bone_Mass_BM', 'Visceral_Fat_Area_VFA', 'Glucose', 'Low_Density_Lipoprotein_LDL', 'High_Density_Lipoprotein_HDL', 'Aspartat_Aminotransferaz_AST', 'Creatinine', 'C-Reactive_Protein_CRP', 'Vitamin_D']
New X_train shape: (223, 12)


In [38]:
# 3rd iteration of Hyperparameter tuned model using 10 folds 

dt_tuned_model_3_1 = optimize_tree_model('DT', n_splits=10, X_train=X_train_selected_dt, y_train=y_train, n_repeats=1, n_iter_search=1, scoring='auc')

evaluate_model(dt_tuned_model_3_1, X_valid_selected_dt, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=10)")


Starting Hyperparameter Search for Decision Tree with 10 evaluations...
Search Method: Grid Search
CV Strategy: Stratified KFold (K=10)
Optimizing for: AUC Score
Fitting 10 folds for each of 1764 candidates, totalling 17640 fits
Grid Search with 10 evaluations complete.

Decision Tree Results
Best AUC Score found: 0.7864

Hyperparameter tuned Decision Tree Model (CV=10) Hyperparameters
  max_depth: 7
  min_samples_leaf: 5
  min_samples_split: 20
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=10) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.75      0.75      0.75        24
   Gallstone       0.75      0.75      0.75        24

    accuracy                           0.75        48
   macro avg       0.75      0.75      0.75        48
weighted avg       0.75      0.75      0.75        48

Accuracy: 75.00%
AUC Score: 83.94%
Log Loss (Calibration): 1.9311
Specificity (True Negative Rate

In [40]:
# 3rd iteration of Hyperparameter tuned model using 10 folds 

dt_tuned_model_3_2 = optimize_tree_model('DT', n_splits=10, X_train=X_train_selected_dt, y_train=y_train, n_repeats=1, n_iter_search=1, scoring='auc')

evaluate_model(dt_tuned_model_3_2, X_valid_selected_dt, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=10)")


Starting Hyperparameter Search for Decision Tree with 10 evaluations...
Search Method: Grid Search
CV Strategy: Stratified KFold (K=10)
Optimizing for: AUC Score
Fitting 10 folds for each of 1764 candidates, totalling 17640 fits
Grid Search with 10 evaluations complete.

Decision Tree Results
Best AUC Score found: 0.7929

Hyperparameter tuned Decision Tree Model (CV=10) Hyperparameters
  max_depth: 4
  min_samples_leaf: 10
  min_samples_split: 25
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=10) Performance
Classification Report:
               precision    recall  f1-score   support

No Gallstone       0.70      0.79      0.75        24
   Gallstone       0.76      0.67      0.71        24

    accuracy                           0.73        48
   macro avg       0.73      0.73      0.73        48
weighted avg       0.73      0.73      0.73        48

Accuracy: 72.92%
AUC Score: 80.21%
Log Loss (Calibration): 1.2218
Specificity (True Negative Rat

In [12]:
import joblib
import os

# Saving the trained models to directory, results are presented in a separate notebook.

TARGET_DIR = r"C:\Python DAT540\Dat-540-Prosjekt\data\tree_models"

# Create the directory if it doesn't exist
if not os.path.exists(TARGET_DIR):
    os.makedirs(TARGET_DIR)
    print(f"Created directory: {TARGET_DIR}")

# Define the models list and a corresponding list of clean filenames
models_to_save = [
    dt_baseline_model, dt_tuned_model_1, dt_tuned_model_2, dt_tuned_model_3,
    dt_tuned_model_4, dt_tuned_model_5, dt_tuned_model_6
]

model_filenames = [
    'dt_baseline_model.joblib',
    'dt_tuned_model_1_CV3.joblib',
    'dt_tuned_model_2_CV5.joblib',
    'dt_tuned_model_3_CV10.joblib',
    'dt_tuned_model_4_CV3_3.joblib',
    'dt_tuned_model_5_CV3_5.joblib',
    'dt_tuned_model_6_CV5_5.joblib'
]

# Loop through and save each model
print("\nSaving models...")
for model, filename in zip(models_to_save, model_filenames):
    
    # os.path.join constructs the full, correct path: C:\...\tree_models\filename.joblib
    full_path = os.path.join(TARGET_DIR, filename) 
    
    joblib.dump(model, full_path)
    print(f"Saved: {full_path}")

print("\nAll models saved successfully.")


Saving models...
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_baseline_model.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_1_CV3.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_2_CV5.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_3_CV10.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_4_CV3_3.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_5_CV3_5.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_6_CV5_5.joblib

All models saved successfully.
